# Task 2: multi-head self-attention with model-wide variables

See **Task 2** on the task sheet for the specification. Implement the model, then run the check cell at the end. No training is needed.

`d_model` and `num_heads` play the role of the model-wide variables: the check cell builds the model with `(d_model, num_heads)` = (32, 4), (64, 8) and (32, 2), and checks the attention scale `1 / sqrt(d_model / num_heads)` in every head.

In [ ]:
import math
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import study_checks
except ImportError:  # running inside the repository
    for rel in ("../../../tools/expert_study", "../../../../tools/expert_study"):
        path = os.path.abspath(rel)
        if os.path.isdir(path):
            sys.path.append(path)
    import study_checks

study_checks.start_session("T2")
x, y = study_checks.load_batch()
print("x", tuple(x.shape), x.dtype, "| y", tuple(y.shape), y.dtype)

In [ ]:
# Allowed: nn.Linear, nn.ReLU / torch.relu, nn.Flatten / torch.flatten, @ / torch.matmul,
#   transpose, scalar multiplication, softmax, torch.cat, +, nn.ModuleList, Python loops,
#   view / reshape (to split heads), F.cross_entropy / nn.CrossEntropyLoss.
# Forbidden: nn.MultiheadAttention, F.scaled_dot_product_attention,
#   F.multi_head_attention_forward, nn.Transformer*.

class Head(nn.Module):
    def __init__(self, d_model, d_head):
        super().__init__()
        self.q = nn.Linear(d_model, d_head)
        self.k = nn.Linear(d_model, d_head)
        self.v = nn.Linear(d_model, d_head)
        self.scale = 1 / math.sqrt(d_head)

    def forward(self, h):
        scores = self.q(h) @ self.k(h).transpose(-2, -1) * self.scale
        return torch.softmax(scores, dim=-1) @ self.v(h)


class MultiHeadRowAttentionClassifier(nn.Module):
    def __init__(self, d_model=32, num_heads=4):
        super().__init__()
        self.embed = nn.Linear(28, d_model)
        self.classifier = nn.Linear(28 * d_model, 10)
        self.heads = nn.ModuleList(Head(d_model, d_model // num_heads) for _ in range(num_heads))

    def forward(self, x):
        h = self.embed(torch.flatten(x, 1, 2))
        attn = torch.cat([head(h) for head in self.heads], dim=-1)
        h = h + attn
        return self.classifier(torch.flatten(h, 1))

In [ ]:
def compute_loss(logits, target):
    return F.cross_entropy(logits, target)  # provided

In [ ]:
study_checks.check_task2(MultiHeadRowAttentionClassifier, compute_loss);